This code is designed to split the LifeSnaps dataset into training and testing participant sets. The final held-out set will contain 20% of participants in the full dataset. We will ensure that the held-out 20% of participants (1) have representative numbers of good and bad mood reports and (2) are not outliers on key sleep, and exercise, and stress metrics : nightly sleep and daily steps, and stress score. 

In [2]:
# load data 
import pandas as pd 
df = pd.read_csv('../../data/processed/aggregate_data_20261027.csv')

/var/folders/j1/0nq47s453531d1krnhqky8sc0000gn/T/ipykernel_58934/621021981.py:3: DtypeWarning: Columns (0: mindfulness_session, 1: bmi, 2: step_goal) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../../data/processed/aggregate_data_20261027.csv')


In [4]:
df.columns

Index(['Unnamed: 0', 'id', 'date', 'hour', 'temperature', 'badgeType',
       'calories', 'distance', 'activityType', 'bpm', 'mindfulness_session',
       'scl_avg', 'steps', 'minutes_in_default_zone_1',
       'minutes_below_default_zone_1', 'minutes_in_default_zone_2',
       'minutes_in_default_zone_3', 'age', 'gender', 'bmi', 'step_goal',
       'min_goal', 'max_goal', 'step_goal_label', 'ALERT', 'HAPPY', 'NEUTRAL',
       'RESTED/RELAXED', 'SAD', 'TENSE/ANXIOUS', 'TIRED', 'ENTERTAINMENT',
       'GYM', 'HOME', 'HOME_OFFICE', 'OTHER', 'OUTDOORS', 'TRANSIT',
       'WORK/SCHOOL', 'BAD_MOOD', 'GOOD_MOOD', 'timestamp',
       'time_since_badmood', 'time_since_goodmood', 'time_until_badmood',
       'time_until_goodmood', 'daily_nightly_temperature', 'daily_nremhr',
       'daily_rmssd', 'daily_spo2', 'daily_full_sleep_breathing_rate',
       'daily_stress_score', 'daily_sleep_points_percentage',
       'daily_exertion_points_percentage',
       'daily_responsiveness_points_percentage'

In [6]:
# Find each subject's daily stress score, distance, and daily sleep duration median. 
median_by_id = (
    df.groupby('id', as_index=False)[
        ['daily_stress_score', 'distance', 'daily_sleep_duration']
    ]
    .median()
)

median_by_id

,id,daily_stress_score,distance,daily_sleep_duration
0,621e2e8e67b776a24055b564,77.0,39.70,33240000.0
1,621e2eaf67b776a2406b14ac,79.0,184.90,28620000.0
2,621e2ed667b776a24085d8d1,NaN,32.20,28980000.0
3,621e2ef567b776a24099f889,NaN,163.75,31140000.0
4,621e2efa67b776a2409dd1c3,NaN,77.00,29100000.0
...,...,...,...,...
66,621e36c267b776a240ba2756,79.0,215.30,23640000.0
67,621e36dd67b776a240ce9a45,0.0,191.10,4260000.0
68,621e36f967b776a240e5e7c9,80.0,111.45,28920000.0
69,621e375367b776a24021e950,NaN,146.70,15720000.0


In [8]:
# calculate distance per-day 
df['distance_perday'] = (
    df.groupby(['id', 'date'])['distance']
      .transform(lambda values: values.sum(min_count=1))
)

In [7]:
# how many men/women? 
df.groupby('gender')['id'].nunique().reindex(['FEMALE', 'MALE', 'OTHER'], fill_value=0)

gender
FEMALE    27
MALE      42
OTHER      0
Name: id, dtype: int64

In [10]:
# make the splits 
from sklearn.model_selection import GroupShuffleSplit

## 80/20 train-test split, keeping each participant entirely in one set
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=18)
train_index, test_index = next(gss.split(df, groups=df["id"]))

df_train = df.iloc[train_index]
df_test = df.iloc[test_index]

In [12]:
# check the split 
print(f"Unique participants in df_train: {df_train['id'].nunique()}")
print(f"Unique participants in df_test: {df_test['id'].nunique()}")


Unique participants in df_train: 56
Unique participants in df_test: 15


In [13]:
cols = ['daily_stress_score', 'distance', 'daily_sleep_duration']  # adjust to your names

print("test share of rows:", len(df_test) / len(df))
pd.concat({"train": df_train[cols].describe().T,
           "test": df_test[cols].describe().T}, axis=1)

test share of rows: 0.23843318203475689


train                                         \
                        count          mean           std        min   
daily_stress_score    33985.0  6.569051e+01  2.653406e+01        0.0   
distance              74520.0  2.890378e+02  5.439279e+02        0.0   
daily_sleep_duration  68666.0  2.719542e+07  6.736669e+06  3600000.0   

                                                                        \
                             25%         50%           75%         max   
daily_stress_score          68.0        75.0  8.000000e+01        94.0   
distance                     4.5        80.5  3.259250e+02     10460.3   
daily_sleep_duration  23820000.0  27600000.0  3.138000e+07  74460000.0   

                         test                                         \
                        count          mean           std        min   
daily_stress_score    10945.0  5.927108e+01  3.348488e+01        0.0   
distance              18613.0  3.422071e+02  5.711882e+02        0.0   
daily_sleep_duration  16418.0  2.619693e+07  6.868880e+06  3720000.0   

                                                                      
                             25%         50%         75%         max  
daily_stress_score          63.0        75.0        81.0        91.0  
distance                     5.1        85.3       434.9      6931.7  
daily_sleep_duration  22560000.0  26580000.0  30420000.0  60000000.0

In [14]:
df_train

,Unnamed: 0,id,date,hour,temperature,badgeType,calories,distance,activityType,bpm,...,daily_minutesToFallAsleep,daily_minutesAsleep,daily_minutesAwake,daily_minutesAfterWakeup,daily_sleep_efficiency,daily_sleep_deep_ratio,daily_sleep_wake_ratio,daily_sleep_light_ratio,daily_sleep_rem_ratio,distance_perday
0,0,621e2e8e67b776a24055b564,2021-05-24,0.0,-1.466659,NaN,89.04,98.3,NaN,66.874763,...,0.0,445.0,76.0,0.0,93.0,1.243243,0.987013,0.921642,1.341772,6517.5
1,1,621e2e8e67b776a24055b564,2021-05-24,1.0,0.210289,NaN,65.62,0.0,NaN,58.711364,...,0.0,445.0,76.0,0.0,93.0,1.243243,0.987013,0.921642,1.341772,6517.5
2,2,621e2e8e67b776a24055b564,2021-05-24,2.0,-1.359378,NaN,65.40,NaN,NaN,55.133495,...,0.0,445.0,76.0,0.0,93.0,1.243243,0.987013,0.921642,1.341772,6517.5
3,3,621e2e8e67b776a24055b564,2021-05-24,3.0,-0.316295,NaN,69.01,11.0,NaN,60.488613,...,0.0,445.0,76.0,0.0,93.0,1.243243,0.987013,0.921642,1.341772,6517.5
4,4,621e2e8e67b776a24055b564,2021-05-24,4.0,0.416455,NaN,65.73,0.0,NaN,59.110638,...,0.0,445.0,76.0,0.0,93.0,1.243243,0.987013,0.921642,1.341772,6517.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159503,159082,621e375b67b776a240290cdc,2021-08-17,7.0,NaN,NaN,54.00,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
159504,159083,621e375b67b776a240290cdc,2021-08-17,8.0,NaN,NaN,54.00,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
159505,159084,621e375b67b776a240290cdc,2021-08-17,9.0,NaN,NaN,54.00,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
159506,159085,621e375b67b776a240290cdc,2021-08-17,10.0,NaN,NaN,54.00,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# make a plot showing 

In [ ]:
from datetime import date

today = date.today().strftime("%Y%m%d")
output_dir = "../../data/processed"

df_train.to_csv(f"{output_dir}/df_train{today}.csv", index=False)
df_test.to_csv(f"{output_dir}/df_test{today}.csv", index=False)